# LeetCode 243 — Shortest Word Distance bằng Python

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

- Link đề: https://leetcode.com/problems/shortest-word-distance/
- Độ khó: **Easy** · Chủ đề: Array, String, duyệt một lần (one pass)

Chữ ký hàm LeetCode:

```python
class Solution:
    def shortestDistance(self, wordsDict: List[str], word1: str, word2: str) -> int:

```

**Mục tiêu học**

1. Hiểu "khoảng cách" giữa hai từ là hiệu **chỉ số (index)** trong mảng, không phải hiệu chữ cái.
2. Ôn `enumerate`, `abs`, `min`, `float('inf')`, list chỉ số.
3. Viết baseline thử mọi cặp vị trí (`O(n²)`), rồi cải tiến bằng **hai danh sách index + hai con trỏ**.
4. Đạt lời giải chính: **một lần duyệt, nhớ vị trí gần nhất** của mỗi từ — `O(n)` thời gian, `O(1)` bộ nhớ.
5. Hiểu vì sao "chỉ cần vị trí gần nhất" là đủ (lập luận tham lam) và chuyển sang biến thể 244/245.

## 1. Đề bài và ví dụ

Cho mảng chuỗi `wordsDict` và hai chuỗi **khác nhau** `word1`, `word2` **đều có mặt** trong mảng. Trả về **khoảng cách ngắn nhất** giữa hai từ này trong danh sách, tức `min |i - j|` với `wordsDict[i] == word1` và `wordsDict[j] == word2`.

Ví dụ với `wordsDict = ["practice", "makes", "perfect", "coding", "makes"]`:

| `word1` | `word2` | Kết quả | Giải thích |
|---|---|---|---|
| `"coding"` | `"practice"` | `3` | coding ở index 3, practice ở 0 → `|3 - 0| = 3` |
| `"makes"` | `"coding"` | `1` | makes ở {1, 4}, coding ở 3 → `min(|1-3|, |4-3|) = 1` |

**Constraints (theo hiểu biết về đề):**

- `2 <= wordsDict.length <= 3 * 10^4`
- `1 <= wordsDict[i].length <= 10`, chỉ gồm chữ thường tiếng Anh
- `word1` và `word2` đều có trong `wordsDict`
- `word1 != word2`

In [1]:
words = ["practice", "makes", "perfect", "coding", "makes"]
for i, w in enumerate(words):
    print(i, w)

0 practice
1 makes
2 perfect
3 coding
4 makes


## 2. Nền tảng Python

### 2.1. `enumerate` — duyệt kèm index

`for i, w in enumerate(words)` cho lần lượt `(0, "practice")`, `(1, "makes")`, ... Ta cần index vì khoảng cách được đo bằng index.

### 2.2. `abs`, `min` và "vô cực"

- `abs(x)`: giá trị tuyệt đối, `abs(1 - 3) = 2`.
- `min(a, b)`: số nhỏ hơn.
- Khởi tạo đáp án bằng `float('inf')` (dương vô cực) để lần so sánh đầu tiên luôn cập nhật. Có thể dùng `len(words)` vì khoảng cách không bao giờ vượt quá `n - 1`.

In [2]:
print(abs(1 - 3), abs(4 - 3))
best = float("inf")
for d in [5, 2, 7]:
    best = min(best, d)
    print("sau khi xét", d, "-> best =", best)
print(float("inf") > 10**18)

2 1
sau khi xét 5 -> best = 5
sau khi xét 2 -> best = 2
sau khi xét 7 -> best = 2
True


### 2.3. List comprehension có điều kiện — lấy mọi index của một từ

`[i for i, w in enumerate(words) if w == "makes"]` = các index mà từ bằng `"makes"`.

In [3]:
idx_makes = [i for i, w in enumerate(words) if w == "makes"]
idx_coding = [i for i, w in enumerate(words) if w == "coding"]
print("makes ở:", idx_makes, "| coding ở:", idx_coding)

makes ở: [1, 4] | coding ở: [3]


## 3. Cách 1 — Brute force: thử mọi cặp vị trí

Ý tưởng tự nhiên nhất: với mỗi `i` mà `words[i] == word1`, với mỗi `j` mà `words[j] == word2`, tính `|i - j|`, lấy min.

In [4]:
def shortest_bruteforce(words: list[str], word1: str, word2: str) -> int:
    best = float("inf")
    for i in range(len(words)):
        if words[i] != word1:
            continue
        for j in range(len(words)):
            if words[j] == word2:
                best = min(best, abs(i - j))
    return best

print(shortest_bruteforce(words, "coding", "practice"))   # 3
print(shortest_bruteforce(words, "makes", "coding"))      # 1

3
1


`continue` bỏ qua phần còn lại của vòng hiện tại, nhảy sang `i` kế tiếp.

### Nút thắt

Trường hợp xấu (nửa mảng là `word1`, nửa là `word2`): `n/2 × n` cặp → `O(n²)`. Với `n = 3·10^4` là ~4.5×10^8 phép so sánh — quá chậm.

In [5]:
def count_pairs_bruteforce(words, w1, w2):
    return sum(1 for i in range(len(words)) if words[i] == w1 for j in range(len(words)))

worst = ["a", "b"] * 500
print("n =", len(worst), "-> số cặp (i, j) phải xét:", count_pairs_bruteforce(worst, "a", "b"))

n = 1000 -> số cặp (i, j) phải xét: 500000


## 4. Cách 2 — Hai danh sách index + hai con trỏ

Thu index của mỗi từ thành hai list **đã tăng dần** (vì ta duyệt từ trái sang). Sau đó dùng hai con trỏ như khi trộn (merge) hai mảng đã sắp xếp:

- So `A[p]` và `B[q]`, cập nhật `best = min(best, |A[p] - B[q]|)`.
- **Tăng con trỏ đang chỉ số nhỏ hơn**: vì giữ nó và tăng con trỏ kia chỉ làm khoảng cách lớn thêm.

In [6]:
def shortest_two_lists(words: list[str], word1: str, word2: str) -> int:
    A = [i for i, w in enumerate(words) if w == word1]
    B = [i for i, w in enumerate(words) if w == word2]
    p = q = 0
    best = float("inf")
    while p < len(A) and q < len(B):
        best = min(best, abs(A[p] - B[q]))
        if A[p] < B[q]:
            p += 1
        else:
            q += 1
    return best

print(shortest_two_lists(words, "coding", "practice"), shortest_two_lists(words, "makes", "coding"))

3 1


Time `O(n)` (tạo list + mỗi con trỏ đi tối đa hết list của nó), Space `O(n)` cho `A`, `B`. Còn có thể bớt bộ nhớ không?

## 5. Insight — chỉ cần nhớ **lần gần nhất** của mỗi từ

Duyệt từ trái sang phải. Khi đang đứng ở index `i` gặp `word1`, cặp tốt nhất **kết thúc tại `i`** với một `word2` phía trước chính là `word2` **gần nhất bên trái** — tức lần cuối cùng ta thấy `word2`. Những `word2` xa hơn ở trái chắc chắn cho khoảng cách lớn hơn.

Mọi cặp `(i, j)` đều có một phần tử đứng sau; khi duyệt tới phần tử đứng sau đó, ta đã so nó với "người gần nhất bên trái" loại kia. Vậy không bỏ sót cặp tối ưu.

Do đó chỉ cần hai biến:

- `pos1` = index gần nhất đã thấy `word1` (khởi tạo `-1` = chưa thấy)
- `pos2` = index gần nhất đã thấy `word2`

In [7]:
pos1, pos2 = -1, -1
for i, w in enumerate(words):
    if w == "makes":
        pos1 = i
    elif w == "coding":
        pos2 = i
    print(f"i={i} {w:<9} pos1(makes)={pos1:>2}  pos2(coding)={pos2:>2}")

i=0 practice  pos1(makes)=-1  pos2(coding)=-1
i=1 makes     pos1(makes)= 1  pos2(coding)=-1
i=2 perfect   pos1(makes)= 1  pos2(coding)=-1
i=3 coding    pos1(makes)= 1  pos2(coding)= 3
i=4 makes     pos1(makes)= 4  pos2(coding)= 3


## 6. Lời giải chính — một lần duyệt

1. `pos1 = pos2 = -1`, `best = len(words)` (lớn hơn mọi khoảng cách có thể).
2. Với mỗi `i, w`:
   - nếu `w == word1`: `pos1 = i`;
   - nếu `w == word2`: `pos2 = i`;
   - nếu cả hai đã từng thấy (`pos1 != -1 and pos2 != -1`): `best = min(best, abs(pos1 - pos2))`.
3. Trả `best`.

Chỉ cập nhật `best` khi vừa gặp một trong hai từ là đủ, nhưng cập nhật mỗi vòng cũng không sai (giá trị không đổi thì min không đổi).

In [8]:
def shortest_one_pass(words: list[str], word1: str, word2: str) -> int:
    pos1 = pos2 = -1
    best = len(words)
    for i, w in enumerate(words):
        if w == word1:
            pos1 = i
        elif w == word2:
            pos2 = i
        else:
            continue                        # từ khác: không có gì thay đổi
        if pos1 != -1 and pos2 != -1:
            best = min(best, abs(pos1 - pos2))
    return best

print(shortest_one_pass(words, "coding", "practice"), shortest_one_pass(words, "makes", "coding"))

3 1


## 7. Trace từng bước

`words = ["a", "c", "b", "x", "a", "b", "c", "a"]`, `word1 = "a"`, `word2 = "b"`:

| i | w | pos1 | pos2 | `|pos1-pos2|` | best |
|---|---|---|---|---|---|
| 0 | a | 0 | -1 | – (chưa đủ) | 8 |
| 1 | c | 0 | -1 | bỏ qua | 8 |
| 2 | b | 0 | 2 | 2 | 2 |
| 3 | x | 0 | 2 | bỏ qua | 2 |
| 4 | a | 4 | 2 | 2 | 2 |
| 5 | b | 4 | 5 | **1** | **1** |
| 6 | c | 4 | 5 | bỏ qua | 1 |
| 7 | a | 7 | 5 | 2 | 1 |

In [9]:
def trace_shortest(words, word1, word2):
    pos1 = pos2 = -1
    best = len(words)
    print(f"{'i':>2} {'w':<5} {'pos1':>4} {'pos2':>4}  hành động")
    for i, w in enumerate(words):
        if w == word1:
            pos1 = i
        elif w == word2:
            pos2 = i
        else:
            print(f"{i:>2} {w:<5} {pos1:>4} {pos2:>4}  từ khác -> bỏ qua")
            continue
        if pos1 != -1 and pos2 != -1:
            d = abs(pos1 - pos2)
            old = best
            best = min(best, d)
            note = f"d={d}, best {old} -> {best}" if best != old else f"d={d}, best giữ {best}"
        else:
            note = "chưa thấy đủ cả hai từ"
        print(f"{i:>2} {w:<5} {pos1:>4} {pos2:>4}  {note}")
    print("Kết quả:", best)
    return best

trace_shortest(["a", "c", "b", "x", "a", "b", "c", "a"], "a", "b")

 i w     pos1 pos2  hành động
 0 a        0   -1  chưa thấy đủ cả hai từ
 1 c        0   -1  từ khác -> bỏ qua
 2 b        0    2  d=2, best 8 -> 2
 3 x        0    2  từ khác -> bỏ qua
 4 a        4    2  d=2, best giữ 2
 5 b        4    5  d=1, best 2 -> 1
 6 c        4    5  từ khác -> bỏ qua
 7 a        7    5  d=2, best giữ 1
Kết quả: 1


1

Chú ý hàng `i=4`: gặp `a` mới, ta "quên" `a` cũ ở index 0 — đúng, vì mọi `b` sau này đều gần `a` ở 4 hơn `a` ở 0.

## 8. Code nộp LeetCode

In [10]:
from typing import List

class Solution:
    def shortestDistance(self, wordsDict: List[str], word1: str, word2: str) -> int:
        pos1 = pos2 = -1
        best = len(wordsDict)
        for i, w in enumerate(wordsDict):
            if w == word1:
                pos1 = i
            elif w == word2:
                pos2 = i
            else:
                continue
            if pos1 != -1 and pos2 != -1:
                best = min(best, abs(pos1 - pos2))
        return best

sol = Solution()
print(sol.shortestDistance(["practice", "makes", "perfect", "coding", "makes"], "coding", "practice"))
print(sol.shortestDistance(["practice", "makes", "perfect", "coding", "makes"], "makes", "coding"))

3
1


`from typing import List` cần cho type hint `List[str]` (LeetCode tự import sẵn; notebook phải tự import).

## 9. Vì sao đúng?

**Invariant:** sau khi xử lý index `i`, `best` = khoảng cách nhỏ nhất trong mọi cặp `(a, b)` với `a, b ≤ i`, `words[a] == word1`, `words[b] == word2`; và `pos1`, `pos2` là index lớn nhất `≤ i` của mỗi từ.

- Xét cặp tối ưu `(a, b)`, giả sử `a < b` (trường hợp kia đối xứng). Tại bước `i = b`, `pos2 = b` và `pos1` là `word1` gần nhất bên trái `b`, tức `pos1 ≥ a`. Do đó `|pos1 - b| ≤ |a - b|` → `best` được cập nhật thành giá trị ≤ tối ưu, mà `best` luôn là khoảng cách của một cặp hợp lệ → `best` = tối ưu.
- Vì `word1 != word2`, mỗi từ chỉ rơi vào đúng một nhánh `if/elif`.

## 10. Độ phức tạp

`n = len(wordsDict)`, `L` = độ dài từ tối đa (≤ 10) — so sánh chuỗi tốn `O(L)`.

| Cách | Time | Space phụ |
|---|---|---|
| Brute force mọi cặp | `O(n² · L)` | `O(1)` |
| Hai list index + hai con trỏ | `O(n · L)` | `O(n)` |
| **Một lần duyệt, nhớ vị trí gần nhất** | `O(n · L)` | `O(1)` |

Vì `L ≤ 10` là hằng, thường viết gọn `O(n)`.

## 11. Bộ test

In [11]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
W = ["practice", "makes", "perfect", "coding", "makes"]
check("ví dụ 1", sol.shortestDistance(W, "coding", "practice"), 3)
check("ví dụ 2", sol.shortestDistance(W, "makes", "coding"), 1)
check("hai phần tử", sol.shortestDistance(["a", "b"], "b", "a"), 1)
check("kề nhau ở cuối", sol.shortestDistance(["a", "x", "x", "x", "a", "b"], "a", "b"), 1)
check("xa nhất", sol.shortestDistance(["a"] + ["x"] * 8 + ["b"], "a", "b"), 9)
check("word2 đứng trước word1", sol.shortestDistance(["b", "x", "a"], "a", "b"), 2)
check("lặp nhiều", sol.shortestDistance(["a", "a", "a", "b", "b", "b"], "a", "b"), 1)

PASS  ví dụ 1: 3
PASS  ví dụ 2: 1
PASS  hai phần tử: 1
PASS  kề nhau ở cuối: 1
PASS  xa nhất: 9
PASS  word2 đứng trước word1: 2
PASS  lặp nhiều: 1


In [12]:
import random
rng = random.Random(243)
solvers = [shortest_bruteforce, shortest_two_lists, shortest_one_pass, Solution().shortestDistance]
for _ in range(2000):
    n = rng.randint(2, 15)
    arr = [rng.choice("abcd") for _ in range(n)]
    i, j = rng.sample(range(n), 2)
    arr[i], arr[j] = "a", "b"                 # đảm bảo cả hai từ có mặt
    outs = [f(arr, "a", "b") for f in solvers]
    assert len(set(outs)) == 1, (arr, outs)
print("2000 mảng ngẫu nhiên: 4 cài đặt khớp nhau — PASS")

2000 mảng ngẫu nhiên: 4 cài đặt khớp nhau — PASS


In [13]:
big = ["x"] * 30000
big[0], big[-1], big[15000], big[15003] = "a", "b", "b", "a"
check("n = 3*10^4", Solution().shortestDistance(big, "a", "b"), 3)

PASS  n = 3*10^4: 3


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ dùng vị trí **đầu tiên** của mỗi từ (`list.index`)

In [14]:
def wrong_index(words, w1, w2):
    return abs(words.index(w1) - words.index(w2))

print("sai :", wrong_index(W, "makes", "coding"), "(phải 1)")

sai : 2 (phải 1)


**Vì sao sai:** `list.index` chỉ trả lần xuất hiện **đầu tiên**; `makes` còn ở index 4 gần `coding` hơn. **Sửa:** xét mọi lần xuất hiện (cách một lần duyệt).

### Lỗi 2: Cập nhật `best` khi một trong hai vị trí còn là `-1`

In [15]:
def wrong_no_guard(words, w1, w2):
    p1 = p2 = -1
    best = len(words)
    for i, w in enumerate(words):
        if w == w1: p1 = i
        elif w == w2: p2 = i
        best = min(best, abs(p1 - p2))       # thiếu kiểm tra -1
    return best

print("sai :", wrong_no_guard(["x", "a", "x", "x", "x", "b"], "a", "b"), "(phải 4)")

sai : 0 (phải 4)


**Vì sao sai:** ngay ở `i=0`, cả hai đều `-1` → `|-1 - (-1)| = 0` được tính như một khoảng cách thật; sau đó ở `i=1` lại có `|1 - (-1)| = 2`. Các giá trị "ảo" này nhỏ hơn đáp án `4`. **Sửa:** chỉ cập nhật khi `p1 != -1 and p2 != -1`.

### Lỗi 3: Khởi tạo `best = 0`
`min(0, ...)` luôn là `0` → luôn trả 0. Phải khởi tạo bằng số **lớn**: `len(words)` hoặc `float('inf')`.

### Lỗi 4: Dùng `if ... if ...` khi đề biến thể cho `word1 == word2`
Ở bài 243, `word1 != word2` nên `if/elif` an toàn. Ở biến thể 245 (hai từ có thể trùng) logic này sai — xem mục biến thể.

## 13. Mẹo nhận dạng pattern

- "Khoảng cách nhỏ nhất giữa hai loại phần tử trong mảng" → **duyệt một lần, nhớ vị trí gần nhất** của mỗi loại.
- "Hai danh sách đã sắp xếp, tìm cặp gần nhau nhất" → **hai con trỏ, tăng con trỏ nhỏ hơn**.
- Nếu phải trả lời **nhiều truy vấn** trên cùng mảng → tiền xử lý `dict: từ -> list index` (biến thể 244).

## 14. Biến thể

### 244 — Shortest Word Distance II (thiết kế, nhiều truy vấn)

Tiền xử lý một lần `dict` từ → danh sách index; mỗi truy vấn dùng hai con trỏ `O(|A| + |B|)`.

In [16]:
from collections import defaultdict

class WordDistance:
    def __init__(self, wordsDict: List[str]):
        self.pos = defaultdict(list)
        for i, w in enumerate(wordsDict):
            self.pos[w].append(i)

    def shortest(self, word1: str, word2: str) -> int:
        A, B = self.pos[word1], self.pos[word2]
        p = q = 0
        best = float("inf")
        while p < len(A) and q < len(B):
            best = min(best, abs(A[p] - B[q]))
            if A[p] < B[q]:
                p += 1
            else:
                q += 1
        return best

wd = WordDistance(W)
print(wd.shortest("coding", "practice"), wd.shortest("makes", "coding"))

3 1


### 245 — Shortest Word Distance III (`word1` có thể bằng `word2`)

Khi hai từ giống nhau, ta cần khoảng cách giữa hai **lần xuất hiện liên tiếp** của cùng một từ.

In [17]:
def shortest_word_distance_iii(words, word1, word2):
    best = len(words)
    if word1 == word2:
        prev = -1
        for i, w in enumerate(words):
            if w == word1:
                if prev != -1:
                    best = min(best, i - prev)
                prev = i
        return best
    return shortest_one_pass(words, word1, word2)

print(shortest_word_distance_iii(W, "makes", "makes"), shortest_word_distance_iii(W, "makes", "coding"))

3 1


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay
`words = ["d", "a", "c", "c", "b", "a", "d", "b"]`, `word1 = "a"`, `word2 = "b"`. Lập bảng `pos1`, `pos2`, `best` như mục 7. Đáp án là bao nhiêu?

### Bài 2 — Lấy mọi index của một từ

In [18]:
def exercise_all_indices(words: list[str], target: str) -> list[int]:
    # TODO: enumerate + điều kiện
    pass

# assert exercise_all_indices(["a", "b", "a"], "a") == [0, 2]

### Bài 3 — Tự viết lời giải một lần duyệt

In [19]:
def exercise_shortest(words: list[str], word1: str, word2: str) -> int:
    # TODO 1: pos1 = pos2 = -1, best = len(words)
    # TODO 2: cập nhật vị trí gần nhất
    # TODO 3: cập nhật best khi đã thấy cả hai
    pass

# assert exercise_shortest(["practice", "makes", "perfect", "coding", "makes"], "makes", "coding") == 1

### Bài 4 — Trả về cả cặp index tốt nhất
Trả `(i, j)` (i là index của `word1`, j của `word2`) đạt khoảng cách nhỏ nhất; nếu hòa, lấy cặp tìm thấy **đầu tiên** khi duyệt từ trái sang.

In [20]:
def exercise_best_pair(words: list[str], word1: str, word2: str) -> tuple[int, int]:
    # TODO: giống bài 3 nhưng lưu thêm cặp khi best giảm (dùng <, không dùng <=)
    pass

# assert exercise_best_pair(["practice", "makes", "perfect", "coding", "makes"], "makes", "coding") == (4, 3)

### Bài 5 — Biến thể 245 (hai từ có thể giống nhau)
Tự viết lại, **trong một vòng lặp duy nhất** (gợi ý: khi `word1 == word2`, mỗi lần gặp từ thì "đẩy" vị trí cũ sang `pos1` và vị trí mới vào `pos2`).

In [21]:
def exercise_distance_iii(words: list[str], word1: str, word2: str) -> int:
    # TODO
    pass

# assert exercise_distance_iii(["practice", "makes", "perfect", "coding", "makes"], "makes", "makes") == 3
# assert exercise_distance_iii(["practice", "makes", "perfect", "coding", "makes"], "makes", "coding") == 1

### Bài 6 — Mini challenge: khoảng cách nhỏ nhất giữa **ba** từ
Tìm đoạn `[l, r]` ngắn nhất (độ dài `r - l`) chứa đủ ba từ `w1, w2, w3`. Gợi ý: nhớ vị trí gần nhất của cả ba, đáp án tại bước `i` là `i - min(ba vị trí)`.

In [22]:
def exercise_three_words(words: list[str], w1: str, w2: str, w3: str) -> int:
    # TODO
    pass

# assert exercise_three_words(["a", "x", "x", "b", "c", "a"], "a", "b", "c") == 2

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chỉ quan tâm hàng có a hoặc b | cập nhật best ở i=4, 5, 7 | đáp án 1 (i=5 a, i=4 b) |
| 2 | Mục 2.3 | `[i for i, w in enumerate(words) if ...]` | nhớ so sánh `w == target` |
| 3 | Mục 6 | `if/elif/else continue` | kiểm tra `-1` trước khi `min` |
| 4 | Lưu `pair` cùng lúc với `best` | chỉ đổi khi `d < best` | `pair = (pos1, pos2)` |
| 5 | Khi trùng từ, hai vị trí là "trước" và "hiện tại" | `pos1, pos2 = pos2, i` | khác từ thì như bài 3 |
| 6 | Ba biến vị trí | chỉ tính khi cả ba ≠ -1 | `best = min(best, i - min(p1, p2, p3))` |

## 17. Đáp án tham khảo

Bài 1: i=1 a → pos1=1; i=4 b → pos2=4, best=3; i=5 a → pos1=5, best=1; i=7 b → pos2=7, d=2. **Đáp án 1.**

In [23]:
def answer_all_indices(words, target):
    return [i for i, w in enumerate(words) if w == target]

def answer_shortest(words, word1, word2):
    pos1 = pos2 = -1
    best = len(words)
    for i, w in enumerate(words):
        if w == word1:
            pos1 = i
        elif w == word2:
            pos2 = i
        if pos1 != -1 and pos2 != -1:
            best = min(best, abs(pos1 - pos2))
    return best

def answer_best_pair(words, word1, word2):
    pos1 = pos2 = -1
    best, pair = len(words), None
    for i, w in enumerate(words):
        if w == word1:
            pos1 = i
        elif w == word2:
            pos2 = i
        else:
            continue
        if pos1 != -1 and pos2 != -1 and abs(pos1 - pos2) < best:
            best, pair = abs(pos1 - pos2), (pos1, pos2)
    return pair

def answer_distance_iii(words, word1, word2):
    pos1 = pos2 = -1
    best = len(words)
    same = word1 == word2
    for i, w in enumerate(words):
        if same and w == word1:
            pos1, pos2 = pos2, i
        elif w == word1:
            pos1 = i
        elif w == word2:
            pos2 = i
        else:
            continue
        if pos1 != -1 and pos2 != -1:
            best = min(best, abs(pos1 - pos2))
    return best

def answer_three_words(words, w1, w2, w3):
    p = {w1: -1, w2: -1, w3: -1}
    best = len(words)
    for i, w in enumerate(words):
        if w in p:
            p[w] = i
            if min(p.values()) != -1:
                best = min(best, i - min(p.values()))
    return best

print(answer_shortest(["d", "a", "c", "c", "b", "a", "d", "b"], "a", "b"))
assert answer_all_indices(["a", "b", "a"], "a") == [0, 2]
assert answer_shortest(W, "makes", "coding") == 1
assert answer_best_pair(W, "makes", "coding") == (4, 3)
assert answer_distance_iii(W, "makes", "makes") == 3 and answer_distance_iii(W, "makes", "coding") == 1
assert answer_three_words(["a", "x", "x", "b", "c", "a"], "a", "b", "c") == 2
for _ in range(500):
    arr = [rng.choice("abc") for _ in range(rng.randint(2, 10))]
    if arr.count("a") >= 2:
        assert answer_distance_iii(arr, "a", "a") == shortest_word_distance_iii(arr, "a", "a")
    if "a" in arr and "b" in arr:
        assert answer_shortest(arr, "a", "b") == shortest_bruteforce(arr, "a", "b")
print("Đáp án tham khảo: tất cả test PASS")

1
Đáp án tham khảo: tất cả test PASS


## 18. Template nhớ nhanh + checklist submit

```python
pos1 = pos2 = -1
best = len(words)
for i, w in enumerate(words):
    if w == word1:   pos1 = i
    elif w == word2: pos2 = i
    else:            continue
    if pos1 != -1 and pos2 != -1:
        best = min(best, abs(pos1 - pos2))
return best
```

**Checklist**

- [ ] Khoảng cách tính bằng **index**, dùng `abs`.
- [ ] Xét **mọi** lần xuất hiện, không dùng `list.index`.
- [ ] Không cập nhật `best` khi một vị trí còn `-1`.
- [ ] `best` khởi tạo đủ lớn (`len(words)` hoặc `inf`).
- [ ] `from typing import List` nếu chạy ngoài LeetCode.
- [ ] Nói được vì sao "vị trí gần nhất" là đủ (cặp tối ưu được xét tại phần tử đứng sau).